Noise analysis with our own function that does the Trotter method

Things to analyse 
- Gate noise
- Hardware noise
- Qubit noise
- Bias noise models?

In [7]:
import Hamiltonian as hm
from qiskit import QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.synthesis import SuzukiTrotter
from qiskit.primitives import StatevectorEstimator
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, pauli_error
from qiskit_aer.primitives import Estimator as AerEstimator
import numpy as np
import matplotlib.pyplot as plt

SparsePauliOp(['IIXX', 'IIYY', 'IIZZ', 'IXXI', 'IYYI', 'IZZI', 'XXII', 'YYII', 'ZZII'],
              coeffs=[-1. +0.j, -1. +0.j, -1.1+0.j, -1. +0.j, -1. +0.j, -1.1+0.j, -1. +0.j,
 -1. +0.j, -1.1+0.j])


### Hugos trotter method below (couldnt figure out how to import ipynb file)

In [8]:
# defining a single trotter_step
def trotter_step_1st_order(circuit: QuantumCircuit, H: SparsePauliOp, dt: float):
    """
    Apply a single 1st order trotter step onto a given QuantumCircuit

    Parameters:
    - circuit (QuantumCircuit): The Quantum Circuit on which the trotter step will be applied
    - H (SparsePauliOp): The hamiltonian represented as a sparse Pauli operator
    - dt (float): The step length in time each trotter step will take

    Returns:
    -circuit (QuantumCircuit): The quatum circuit after applying one trotter step
    """
    for pauli_str, coeff in H.to_list():
        evolution_gate = PauliEvolutionGate(pauli_str, time=dt * coeff.real)
        circuit.append(evolution_gate, range(circuit.num_qubits))
    return circuit

# defining a function that applies the single trotter step n times
def first_order_trotter(H: SparsePauliOp, t: float, n_steps: int, num_qubits: int = None) -> QuantumCircuit:
    """
    Apply trotter_step_1st_order onto a quantum circuit n qubits for n_steps to evolve for time t.

    Parameters:
    - H (SparsePauliOp): The hamiltonian represented as a sparse Pauli operator
    - t (float): length of time to evolve qubits for
    - n_steps (int): Number of steps applied to circuit
    - num_qubits (int): Number of qubits in the system

    Return:
    - qc (QuantumCircuit): circuit implemeting time evolution
    """
    if num_qubits is None:
        num_qubits = H.num_qubits
    
    # create circuit with width = num_qubits
    qc = QuantumCircuit(num_qubits)
    # find the length of each individual trotter step
    dt = t/n_steps
    # loop
    for _ in range(n_steps):
        trotter_step_1st_order(qc, H, dt)
    return qc

In [ ]:
def make_noise_model(noise_type='all', p1=0.005, p2=0.02):
    """_summary_

    Args:
        noise_type (str, optional): _description_. Defaults to 'all'.
        p1 (float, optional): _description_. Defaults to 0.005.
        p2 (float, optional): _description_. Defaults to 0.02.
    """
    nm = NoiseModel() #defines the noise we are using

    if noise_type == 'X':
        # bit flip 0 to 1 and 1 to 0
        error = depolarizing_error(p1, 1)
        nm.add_all_qubit_quantum_error(error, ['x', 'h', 'sx', 'ry', 'rz'])

    elif noise_type == 'Z':
        # phase flip 0 to 0 and 1 to -1
        error = pauli_error([('Z', p1), ('I', 1 - p1)])
        nm.add_all_qubit_quantum_error(error, ['x', 'h', 'sx', 'ry', 'rz'])

    elif noise_type == 'all':
        error_1q = depolarizing_error(p1, 1)
        nm.add_all_qubit_quantum_error(error_1q, ['x', 'h', 'sx', 'ry', 'rz'])
        # two gates
        error_2q = depolarizing_error(p2, 2)
        nm.add_all_qubit_quantum_error(error_2q, ['cx', 'cz'])

    else:
        raise ValueError("noise_type must be 'X', 'Z', or 'all'")

    return nm 

def TheoreticalST_noisy(L, Jz, qubit_measured, direction_measured, steps, t_tot,
                         noise_type='all', p1=0.005, p2=0.02,
                         order=1, reps=1, allzeros=True):
    """
    Same as hm.TheoreticalST but runs through a noisy AerEstimator.
    Swap noise_type=None to get the noiseless StatevectorEstimator result for comparison.
    """
    hamiltonian = hm.hamiltonian1(L, Jz)
    dt = t_tot / steps
    gate = PauliEvolutionGate(operator=hamiltonian, time=dt)
    st = SuzukiTrotter(order=order, reps=reps)
    circ = st.synthesize(gate)

    # Build observable string e.g. 'IIXI' for qubit 2, direction X, L=4
    paulistring = ""
    for i in range(L):
        paulistring += direction_measured if i == qubit_measured - 1 else "I"
    obs = SparsePauliOp([paulistring], coeffs=[1])

    # Initial state from your hamiltonian file
    if Jz > 1:
        all0, all1 = hm.initialise(Jz, L)[1]
        initial = all0 if allzeros else all1
    elif Jz < -1:
        initial = hm.initialise(Jz, L)[1][0]
    else:
        raise ValueError('Please pick J_z > 1 or J_z < -1')

    # Rotate middle qubit (same as your original)
    qc = QuantumCircuit(int(L))
    qc.initialize(initial)
    qc.ry(np.pi / 2, L // 2)
    initial = Statevector(qc)
    qc = QuantumCircuit(int(L))
    qc.initialize(initial)

    # Choose estimator
    if noise_type is None:
        estimator = StatevectorEstimator()
        measurements = []
        for _ in range(steps):
            qc.append(circ, list(range(L)))
            result = estimator.run(pubs=[(qc, [obs])]).result()
            measurements.append(result[0].data.evs[0])

    else:
        nm = make_noise_model(noise_type, p1, p2)
        estimator = AerEstimator()
        estimator.set_options(noise_model=nm, shots=10000)
        measurements = []
        for _ in range(steps):
            qc.append(circ, list(range(L)))
            # V1 API: run(circuits, observables) separately, not as pubs
            result = estimator.run([qc], [obs]).result()
            measurements.append(result.values[0]) 
    t_values = np.array([dt * t for t in range(steps)])
    return t_values, np.array(measurements)

def plot_noise_comparison(L, Jz, qubit, direction, steps, t_tot, p1=0.005, p2=0.02):
    """
    Plots noiseless vs X-noise vs Z-noise vs all-noise on the same axes.
    """
    configs = [
        (None,  'Noiseless', 'black'),
        ('X',   'X noise',   'royalblue'),
        ('Z',   'Z noise',   'crimson'),
        ('all', 'All noise', 'forestgreen'),
    ]

    for noise_type, label, color in configs:
        print(f"Running: {label}...")
        t, vals = TheoreticalST_noisy(
            L, Jz, qubit, direction, steps, t_tot,
            noise_type=noise_type, p1=p1, p2=p2
        )
        plt.plot(t, vals, label=label, color=color)

    plt.xlabel(r'$t$ $(eV^{-1})$')
    plt.ylabel(rf'$\langle {direction}_{qubit}(t) \rangle$')
    plt.title(rf'Noise comparison — qubit {qubit}, {direction}-direction, $J_z$={Jz}')
    plt.legend()
    plt.ylim([-1.1, 1.1])
    plt.show()
    plt.close()

L, Jz = 4, 1.1

plot_noise_comparison(L, Jz, qubit=1, direction='Z', steps=80, t_tot=1.5)
plot_noise_comparison(L, Jz, qubit=1, direction='X', steps=80, t_tot=5)
plot_noise_comparison(L, Jz, qubit=1, direction='Y', steps=80, t_tot=5)

Running: Noiseless...
Running: X noise...


C:\Users\laram\AppData\Local\Temp\ipykernel_14232\3613925994.py:104: DeprecationWarning: Estimator has been deprecated as of Aer 0.15, please use EstimatorV2 instead.
  t, vals = TheoreticalST_noisy(
C:\Users\laram\AppData\Local\Temp\ipykernel_14232\3613925994.py:104: DeprecationWarning: Option approximation=False is deprecated as of qiskit-aer 0.13. It will be removed no earlier than 3 months after the release date. Instead, use BackendEstimator from qiskit.primitives.
  t, vals = TheoreticalST_noisy(
